In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


In [17]:

def process_pop_data(pop_data):

    pop_data = pop_data[['ISO3', 'Population_group', 'Population', 'Reference_year']]
    pop_data = pop_data[pop_data['Population_group'] == 'T_TL']
    pop_data = pop_data.loc[pop_data.groupby('ISO3')['Reference_year'].idxmax()]
    pop_data = pop_data.reset_index(drop=True)

    return pop_data

def process_funding_req_data(funding_req_data):

    funding_req_data = funding_req_data[['countryCode', 'year', 'requirements', 'funding']]
    funding_req_data = funding_req_data[funding_req_data['year'] < 2027]
    funding_req_data = funding_req_data.fillna(0)
    funding_req_data = funding_req_data.groupby('countryCode')[['requirements', 'funding']].sum().reset_index()
    funding_req_data["funding_percentage"] = np.where(
        funding_req_data["requirements"] == 0, 
        100.0, 
        (funding_req_data["funding"] / funding_req_data["requirements"]) * 100
    )

    return funding_req_data

def funding_to_population_ratio(funding_req_data, pop_data):

    combined_df = pd.merge(
        funding_req_data,
        pop_data[['ISO3', 'Population']],  # We only pull the relevant columns from pop data
        left_on='countryCode',             # The key column in funding_req_data
        right_on='ISO3',                   # The key column in pop_data
        how='left'                         # Keeps all rows from funding_req_data
    )

    combined_df.drop(columns=['ISO3'], inplace=True)

    combined_df["funding_pct_div_pop"] = np.where(
        combined_df["funding_percentage"] == 0, 
        0, 
        combined_df["Population"] / combined_df["funding_percentage"]
    )

    # Calculate the min and max values of the column
    min_val = combined_df["funding_pct_div_pop"].min()
    max_val = combined_df["funding_pct_div_pop"].max()

    # Apply the Min-Max normalization formula
    # We add a small check (max_val - min_val != 0) to avoid any potential division by zero
    if max_val - min_val != 0:
        combined_df["funding_pct_div_pop_normalized"] = (combined_df["funding_pct_div_pop"] - min_val) / (max_val - min_val)
    else:
        combined_df["funding_pct_div_pop_normalized"] = 0.0


    return combined_df


def create_choropleth_map(combined_df):

    # ---------------------------------------------------------------------------
    # 1. Creating the Plotly Figure
    # ---------------------------------------------------------------------------
    # We define a custom color scale going from Blue (0) to Red (1)
    blue_to_red = [[0.0, 'rgb(44, 165, 141)'], [1.0, 'rgb(4, 14, 26)']]

    fig = px.choropleth(
        combined_df,
        locations="countryCode",               # Column containing ISO3 codes
        color="funding_pct_div_pop_normalized", # Column dictating the color intensity
        hover_name="countryCode",              # Title text inside the hover tooltip
        color_continuous_scale=blue_to_red,    # Custom color transition
        labels={'funding_pct_div_pop_normalized': 'Normalized Ratio'},
        title="Country Funding-to-Population Normalized Map"
    )

    # Customize map appearance & handle the NaN/No Data color
    fig.update_geos(
        showframe=False,
        showcoastlines=True,
        projection_type="equirectangular",
        showcountries=True,                    # Outlines all world countries
        countrycolor="white",                  # Border lines color
        landcolor="#E5E5E5"                    # Sets base/missing data countries to a clean light grey
    )

    fig.update_layout(
        margin={"r":0,"t":50,"l":0,"b":0},
        coloraxis_autocolorscale=False
    )

    return fig


In [15]:
pop_data = pd.read_excel('cod_population_admin0.xlsx')
funding_req_data = pd.read_excel('fts_requirements_funding_global.xlsx')
hpc_data = pd.read_excel('hpc_hno_2026.xlsx')
hrp_data = pd.read_excel('humanitarian_response_plans.xlsx')

pop_data = process_pop_data(pop_data)
funding_req_data = process_funding_req_data(funding_req_data)
crisis_severity_df = funding_to_population_ratio(funding_req_data, pop_data)

In [18]:
crisis_severity_df.to_excel('crisis_severity_analysis.xlsx', index=False)

crisis_map_fig = create_choropleth_map(crisis_severity_df)
crisis_map_fig.show()